# Lab 49: Computacion Cuantica Fotonica

Los **fotones** son portadores de informacion cuantica naturalmente resistentes a la decoherencia. En este lab exploraremos:
- La funcion de Wigner y sus valores negativos como indicador de no-clasicidad.
- Boson Sampling: por que el permanente de una matriz es computacionalmente duro.
- Operaciones gaussianas: squeezing, desplazamiento, haz divisor.
- El codigo GKP: correccion de errores en el oscilador armonico.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import factorial, eval_genlaguerre
from scipy.linalg import expm
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

## 1. Funcion de Wigner — Visualizacion en Fase-Espacio

La funcion de Wigner $W(x, p)$ es la distribucion cuasi-probabilistica de un estado cuantico. Sus valores negativos senalan no-clasicidad.

Para el estado de Fock $|n\rangle$:
$$W_n(x,p) = \frac{(-1)^n}{\pi}\,e^{-2(x^2+p^2)}\,L_n\!\left(4(x^2+p^2)\right)$$
donde $L_n$ son los polinomios de Laguerre.

In [ ]:
def wigner_fock(n, x, p):
    r2 = 2 * (x**2 + p**2)
    return ((-1)**n / np.pi) * np.exp(-r2) * eval_genlaguerre(n, 0, 2*r2)

def wigner_coherent(alpha, x, p):
    return (1/np.pi) * np.exp(-2*((x - alpha.real)**2 + (p - alpha.imag)**2))

def wigner_squeezed(r, x, p):
    return (1/np.pi) * np.exp(-2*(x**2 * np.exp(2*r) + p**2 * np.exp(-2*r)))

N = 200
xv = np.linspace(-4, 4, N)
pv = np.linspace(-4, 4, N)
X, P = np.meshgrid(xv, pv)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
configs = [
    ('|0> Vacio',        wigner_fock(0, X, P),        'Blues'),
    ('|1> 1 foton',      wigner_fock(1, X, P),        'RdBu'),
    ('|2> 2 fotones',    wigner_fock(2, X, P),        'RdBu'),
    ('|alpha=2> Coher.', wigner_coherent(2.0, X, P),  'Blues'),
    ('Squeezed r=1.0',   wigner_squeezed(1.0, X, P),  'Greens'),
    ('Sum |0>+coh(2)',   wigner_fock(0,X,P)+wigner_coherent(2.0,X,P), 'RdBu'),
]
for ax, (title, W, cmap) in zip(axes.flat, configs):
    vmax = max(abs(W.min()), abs(W.max()))
    im = ax.pcolormesh(X, P, W, cmap=cmap, vmin=-vmax, vmax=vmax, shading='auto')
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_xlabel('x'); ax.set_ylabel('p')
    plt.colorbar(im, ax=ax, shrink=0.8)
    if W.min() < -1e-6:
        ax.text(0.04, 0.92, 'W<0: no clasico', transform=ax.transAxes,
                fontsize=8, color='red', fontweight='bold')
plt.suptitle('Funcion de Wigner', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print("Minimos de W (negativos = no clasico):")
for n in range(4):
    wmin = wigner_fock(n, X, P).min()
    print(f"  |{n}>: W_min = {wmin:.4f} {'<-- no clasico' if wmin < 0 else ''}")

## 2. Boson Sampling — El Permanente es #P-Duro

El Boson Sampling muestrea de la distribucion de salida de una red de optica lineal aleatoria $U$ con $n$ fotones de entrada.

$$p(s_1,\ldots,s_m) \propto |\mathrm{Perm}(U_{S,T})|^2$$

El **permanente** de una matriz $n\times n$ es $\#P$-duro: computable en $O(2^n n)$ con Ryser, intractable para $n > 40$ clasicamente.

In [ ]:
def permanent_ryser(A):
    n = A.shape[0]
    total = 0.0 + 0j
    for subset in range(1, 1 << n):
        bits = [j for j in range(n) if subset & (1 << j)]
        row_sums = np.array([sum(A[i, j] for j in bits) for i in range(n)])
        sign = (-1) ** (n - len(bits))
        total += sign * np.prod(row_sums)
    return (-1)**n * total

print("Operaciones del algoritmo de Ryser (~2^n * n):")
for n in [2, 3, 4, 5, 8, 10, 15, 20, 50]:
    ops = 2**n * n
    tag = '' if ops < 1e9 else '  <- intractable clasicamente'
    print(f"  n={n:2d}: {ops:.2e} ops{tag}")

# Distribucion de salida para 3 fotones en 6 modos
from itertools import combinations_with_replacement
from collections import Counter

def haar_unitary(m, seed=0):
    rng = np.random.default_rng(seed)
    Z = rng.standard_normal((m,m)) + 1j*rng.standard_normal((m,m))
    Q, R = np.linalg.qr(Z / np.sqrt(2))
    return Q @ np.diag(R.diagonal()/np.abs(R.diagonal()))

m, n_ph = 6, 3
U = haar_unitary(m, seed=42)
input_modes = [0, 1, 2]

probs = {}
for out in combinations_with_replacement(range(m), n_ph):
    U_sub = U[np.ix_(list(out), input_modes)]
    p = abs(permanent_ryser(U_sub))**2
    in_c = Counter(input_modes); out_c = Counter(out)
    denom = np.prod([factorial(k) for k in in_c.values()]) * np.prod([factorial(k) for k in out_c.values()])
    probs[out] = float(p / denom)

total_p = sum(probs.values())
probs = {k: v/total_p for k, v in probs.items()}
top10 = sorted(probs.items(), key=lambda x: -x[1])[:10]

plt.figure(figsize=(10, 4))
plt.bar([str(k) for k,_ in top10], [v for _,v in top10], color='steelblue', alpha=0.8)
plt.xlabel('Configuracion de salida'); plt.ylabel('Probabilidad')
plt.title(f'Boson Sampling: Top-10 salidas ({n_ph} fotones, {m} modos)')
plt.xticks(rotation=30, fontsize=9); plt.tight_layout(); plt.show()
print(f"Total configuraciones: {len(probs)}, suma: {sum(probs.values()):.6f}")

## 3. Operaciones Gaussianas — Squeezing y Divisor de Haz

Las operaciones gaussianas se representan con matrices simetricticas $S \in Sp(2n, \mathbb{R})$:
$$\mathbf{r} \to S\mathbf{r}$$
El squeezing reduce la varianza en $x$ en un factor $e^{-2r}$ y la amplifica en $p$ en $e^{+2r}$, preservando el producto $\Delta x \cdot \Delta p = 1/2$.

In [ ]:
def squeezing_S(r):
    return np.array([[np.exp(-r), 0], [0, np.exp(r)]])

def beamsplitter_S(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.block([[c*np.eye(2), -s*np.eye(2)],[s*np.eye(2), c*np.eye(2)]])

# Varianzas tras squeezing
r_vals = np.linspace(0, 2, 200)
var_x  = 0.5 * np.exp(-2*r_vals)
var_p  = 0.5 * np.exp(+2*r_vals)
r_dB   = 8.686 * r_vals

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].semilogy(r_dB, var_x, 'steelblue', lw=2, label='Var(X) comprimida')
axes[0].semilogy(r_dB, var_p, 'darkorange', lw=2, label='Var(P) amplificada')
axes[0].axhline(0.5, color='gray', ls=':', label='Vacio')
axes[0].axvline(10, color='red', ls='--', lw=1.5, label='Umbral GKP (~10 dB)')
axes[0].set_xlabel('Squeezing (dB)'); axes[0].set_ylabel('Varianza')
axes[0].set_title('Varianzas tras squeezing'); axes[0].legend(fontsize=9)

heis = var_x * var_p
axes[1].plot(r_dB, heis, 'mediumseagreen', lw=2)
axes[1].axhline(0.25, color='red', ls='--', label='Min Heisenberg')
axes[1].set_xlabel('Squeezing (dB)'); axes[1].set_ylabel('DeltaX * DeltaP')
axes[1].set_title('Producto de incertidumbre = constante'); axes[1].legend()
plt.tight_layout(); plt.show()

# Squeezing en dB para varios valores de r
print("Squeezing en dB:")
for r in [0.5, 1.0, 1.15, 1.5, 2.0]:
    dB = 8.686 * r
    print(f"  r={r:.2f}: {dB:.1f} dB, Var(X)={0.5*np.exp(-2*r):.4f}, Var(P)={0.5*np.exp(2*r):.4f}")
print(f"  Umbral GKP: r >= {np.log(10**(10/8.686)):.3f} (10 dB)")

## 4. Codigo GKP — Correccion de Errores en el Oscilador

El codigo GKP codifica 1 qubit logico en el oscilador armonico. Los estados logicos son 'peines' de gaussianas en espacio-x con periodo $2\sqrt{\pi}$:

$$|0_L\rangle \propto \sum_n |x=2n\sqrt{\pi}\rangle, \quad |1_L\rangle \propto \sum_n |x=(2n+1)\sqrt{\pi}\rangle$$

Un error de desplazamiento $\delta x$ es corregible si $|\delta x| < \sqrt{\pi}/2 \approx 0.89$.

In [ ]:
def gkp_approx(x, logical, Delta=0.25, N_peaks=7):
    step = 2 * np.sqrt(np.pi)
    offset = 0.0 if logical == 0 else np.sqrt(np.pi)
    psi = np.zeros_like(x, dtype=float)
    for n in range(-N_peaks, N_peaks+1):
        center = n * step + offset
        psi += np.exp(-0.5 * ((x - center) / Delta)**2)
    norm = np.sqrt(np.trapz(psi**2, x))
    return psi / norm

x = np.linspace(-9, 9, 3000)
psi0 = gkp_approx(x, 0)
psi1 = gkp_approx(x, 1)
threshold = np.sqrt(np.pi) / 2

delta_small = 0.3
delta_large = 1.2

psi0_err_s = gkp_approx(x - delta_small, 0)
psi0_err_l = gkp_approx(x - delta_large, 0)

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
axes[0,0].fill_between(x, psi0**2, alpha=0.7, color='steelblue')
axes[0,0].set_title('|0_L> GKP', fontsize=11); axes[0,0].set_xlabel('x')

axes[0,1].fill_between(x, psi1**2, alpha=0.7, color='darkorange')
axes[0,1].set_title('|1_L> GKP', fontsize=11); axes[0,1].set_xlabel('x')

for ax, psi_e, delta, label in [
    (axes[1,0], psi0_err_s, delta_small, f'delta={delta_small} (corregible)'),
    (axes[1,1], psi0_err_l, delta_large, f'delta={delta_large} (NO corregible)'),
]:
    ax.fill_between(x, psi0**2, alpha=0.3, color='steelblue', label='|0_L> ideal')
    ax.fill_between(x, psi_e**2, alpha=0.7, color='crimson', label=label)
    ax.axvline(threshold, color='green', ls='--', lw=1.5, label=f'Limite sqrt(pi)/2={threshold:.2f}')
    ax.axvline(-threshold, color='green', ls='--', lw=1.5)
    ax.legend(fontsize=8); ax.set_xlabel('x')

axes[1,0].set_title('Error pequeno — corregible')
axes[1,1].set_title('Error grande — NO corregible')
for ax in axes.flat: ax.set_xlim([-8,8]); ax.set_ylabel('|psi(x)|^2')
plt.suptitle('Codigo GKP: estados logicos y correccion de errores', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

overlap = abs(np.trapz(psi0 * psi1, x))
print(f"Solapamiento |<0_L|1_L>| = {overlap:.2e}")
print(f"Umbral GKP: |delta_x| < sqrt(pi)/2 = {threshold:.4f}")
for d in [delta_small, delta_large]:
    print(f"  delta={d}: {'corregible' if d < threshold else 'NO corregible'}  ({d:.2f} {'<' if d < threshold else '>='} {threshold:.4f})")

## 5. Comparativa de Plataformas Fotonicas

Resumimos las metricas clave de los principales sistemas de computacion fotonica actuales y comparamos con superconductores e iones trampa.

In [ ]:
# Tabla comparativa de plataformas
plataformas = {
    'Sistema':         ['PsiQuantum',   'Xanadu Borealis', 'Quandela QD', 'IBM Heron r2', 'IonQ Forte'],
    'Plataforma':      ['Fotonico DV',  'Fotonico CV',     'Fotonico DV', 'Supercond.',   'Iones'],
    'Error 2Q (%)':    [3.5,            1.0,               2.0,           0.3,            0.1],
    'T_oper.':         ['Ambiente',     'Ambiente',        '4 K',         '15 mK',        'Ultra-vac'],
    'Velocidad gate':  ['~ns',          '~us',             '~ns',         '30 ns',        '600 us'],
}

import pandas as pd
df = pd.DataFrame(plataformas)
print(df.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#4472C4','#4472C4','#4472C4','#FF7F0E','#2CA02C']
for i, (sys, err, c) in enumerate(zip(plataformas['Sistema'], plataformas['Error 2Q (%)'], colors)):
    ax.bar(i, err, color=c, alpha=0.8, edgecolor='black', linewidth=0.5)
    ax.text(i, err + 0.05, f'{err}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.axhline(1.0, color='red', ls='--', lw=2, label='Umbral corr. errores (~1%)')
ax.set_xticks(range(5)); ax.set_xticklabels(plataformas['Sistema'], rotation=15, fontsize=9)
ax.set_ylabel('Error puerta 2Q (%)', fontsize=12)
ax.set_title('Plataformas cuanticas: error 2Q (2025)', fontsize=12)
ax.legend(); ax.set_ylim([0, 5])
from matplotlib.patches import Patch
handles = [Patch(color='#4472C4', label='Fotonico'), 
           Patch(color='#FF7F0E', label='Superconductor'),
           Patch(color='#2CA02C', label='Iones trampa')]
ax.legend(handles=handles + [plt.Line2D([0],[0],color='red',ls='--',lw=2,label='Umbral 1%')], fontsize=9)
plt.tight_layout(); plt.show()

print("\nResumen Fase 19 — Computacion Fotonica:")
print("  - Funcion de Wigner: W<0 indica no-clasicidad (estados Fock, gatos)")
print("  - Boson Sampling: perm es #P-duro, ventaja cuantica demostrada (Jiuzhang 2020-21)")
print("  - Squeezing: necesario >10 dB para GKP tolerante a fallos")
print("  - Codigo GKP: corrige |delta_x| < sqrt(pi)/2 en el oscilador")
print("  - KLM: universal con optica lineal + PNR + feed-forward")

## Resumen

| Concepto | Resultado clave |
|---|---|
| Funcion de Wigner | $W < 0$ en $|n\rangle$ con $n \geq 1$; gaussiana positiva para estados coherentes |
| Boson Sampling | $p \propto |\mathrm{Perm}(U)|^2$; Ryser escala $O(2^n n)$; ventaja demostrada con 113 fotones |
| Squeezing | $\mathrm{Var}(X) = e^{-2r}/2$; umbral GKP $\approx 10$ dB $(r \approx 1.15)$ |
| Codigo GKP | Corrige $|\delta x| < \sqrt{\pi}/2 \approx 0.89$; solapamiento $|\langle 0_L|1_L\rangle| \approx 0$ |
| KLM | Universal con optica lineal + detectores PNR + feed-forward clasico |
| Estado del hardware | Fotonico: error 2Q $\sim 1\text{-}5\%$; superconductor: $0.1\text{-}0.5\%$; iones: $0.1\%$ |